← [02 · From pixels to a light curve](02_from_pixels_to_light_curve.ipynb) · [Index](README.md) · [04 · False positives](04_false_positives_and_noise.ipynb) →
<!--nav-->

# 03 · Detrending and finding periods

We can now get a clean-ish light curve. Two problems remain before a planet falls out: (1) the curve has slow **trends** that dwarf a transit, and (2) we need to find the period **automatically** instead of knowing it in advance.

**You'll learn:** what systematics/trends are and how **flattening** removes them · the difference between **Lomb–Scargle** and **BLS** periodograms · how to recover a period and confirm it by folding · why the smoothing window is the knob that ruins detections.

This notebook uses `skyplay`, the project's own package, for the steps notebooks 01–02 built up by hand. Each call below says what it wraps, and the source lives in `src/skyplay/` if you want to read it.

This notebook is the direct lead-in to the capstone, [`kepler8b_transit_recovery.ipynb`](kepler8b_transit_recovery.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from skyplay import data, detrend, periods, plotting

plotting.use_style()

# Searches MAST, downloads quarters 1-4, normalizes each to its own median,
# concatenates them and drops NaNs -- exactly what notebook 01 did by hand.
# The result is cached to data/cache/, so re-running this later is instant.
lc = data.load_stitched('kepler-8')

print(f'{len(lc)} cadences over {(lc.time.max() - lc.time.min()).value:.0f} days')
print(f'time format: {lc.time.format} ({lc.time.scale})')
lc.scatter(s=1)
plt.show();

## 1. Detrending (flattening)

Those slow rolls are **stellar variability** (starspots rotating in and out of view) and **instrument systematics** (temperature drifts, pointing changes). On an active star they dwarf a planetary transit. On this fairly quiet one they turn out to be *smaller* than it — the cell below measures both, so you can check rather than take my word for it. That is a useful calibration: the received wisdom that "trends dwarf transits" is true in general and false for this particular star. Either way they have to go, because they bias the depth you measure and they confuse a period search.

We remove them by fitting a smooth trend and dividing it out. `detrend.savgol_flatten` wraps lightkurve's `flatten()`, which does this with a Savitzky–Golay filter.

⚠️ **The one knob that bites people:** the smoothing window must be **longer than a transit**. If it's too short, the filter treats the transit itself as "trend" and irons it flat — deleting the very thing you're looking for. We use ~19 days here, far longer than the few-hour transit, so we're safe. **Section 4 measures exactly what goes wrong when you get this one wrong.**

📐 **A units trap worth internalising:** lightkurve's `flatten(window_length=...)` counts **cadences**, not days — the `901` you'll see in older code and in the capstone is 901 × 30 min ≈ 18.8 days. wotan's equivalent counts **days**. Passing `901` to wotan asks for a 901-*day* window and silently does nothing. `skyplay.detrend` takes **days** in both cases and converts, so you only have to think about this once.

In [ ]:
# window_days=18.8 is the same filter as the capstone's window_length=901 cadences.
flat, trend = detrend.savgol_flatten(lc, window_days=18.8)

# Detrending removes *correlated* structure. It cannot touch the white noise, which
# is why the per-point scatter barely changes.
print(f'trend amplitude removed : {np.ptp(trend.flux.value) * 1e6:5.0f} ppm')
print(f'per-point white noise   : {np.diff(flat.flux.value).std() / np.sqrt(2) * 1e6:5.0f} ppm')
print('(the transit turns out to be ~8400 ppm -- on this star it is ~3x DEEPER than')
print(' the trend, which is why Kepler-8 b was an easy early find)')

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

lc.scatter(ax=axes[0], s=1, label='stitched')
trend.plot(ax=axes[0], color=plotting.SERIES[1], label='Savitzky-Golay trend')
axes[0].set_title('Stitched, with the trend that is about to be divided out')

flat.scatter(ax=axes[1], s=1, label='flattened')
axes[1].set_title('Flattened: trend gone, transits now the only structure left')
plt.show();

Now the curve sits flat at 1.0 and the transits are the dominant feature. (You may even spot the periodic dips by eye.)

## 2. Finding the period: periodograms

A **periodogram** tries many trial periods and scores how strongly the data repeats at each. Two you'll meet:

- **Lomb–Scargle** — looks for *sinusoidal* variations. Great for smooth things: pulsating or spotted stars, rotation. Wrong tool for transits, whose dips are sharp boxes, not sine waves.
- **Box Least Squares (BLS)** — looks specifically for a periodic *box-shaped dip*, spending most of the cycle flat with a brief drop. This is the transit-hunter's tool.

We ask BLS to score periods from 1–10 days. We do **not** tell it the answer — the peak *is* the discovery.

In [ ]:
# We do not pass the answer in -- only a range of periods to score.
bls = periods.bls_search(flat, period_min=1, period_max=10, n_periods=20_000)

print(bls.summary())

# The peak is marked, and P/2 and 2P are flagged: those aliases are where the
# classic false positive hides (an eclipsing binary found at half its true period).
plotting.plot_spectrum(bls)
plt.show();

The single sharp spike is the orbital period the data prefers. (The smaller spikes at multiples/fractions are *aliases* — harmonics of the true period.)

## 3. Confirm by folding

A period from a periodogram is a *hypothesis*. We confirm it the way notebook 00 taught: fold on it and check that a clean, coherent transit appears.

In [ ]:
# Gray points are individual cadences; the blue line is the binned average.
plotting.plot_folded(
    flat.time.value, flat.flux.value, bls.period, bls.epoch, phase_window=0.05
)
plt.show();

## 4. The window knob, measured

Section 1 warned that too short a smoothing window makes the filter iron the transit flat. That's easy to state and easy to ignore, so let's measure it — we have an ephemeris now, so we can re-detrend at a range of window widths and ask how deep the transit still looks.

Kepler-8 b's transit lasts about **3 hours (0.13 days)**, and is really **~8,950 ppm** deep. We measure two versions at each width:

- **clipped** — lightkurve's default behaviour.
- **no clipping** — the same Savitzky–Golay filter with its outlier rejection switched off.

The gap between those two columns is the interesting part.

In [ ]:
from skyplay.detrend import days_to_cadences
from skyplay.vetting import vet

def measured_depth(curve):
    report = vet(curve.time.value, curve.flux.value, bls.period, bls.epoch, halfwidth=0.01)
    return report.transit_depth * 1e6

print(f'{"window (d)":>10s} {"cadences":>9s} {"clipped":>12s} {"no clipping":>14s}')
for window_days in (18.8, 2.0, 0.5, 0.25, 0.13, 0.08):
    clipped, _ = detrend.savgol_flatten(lc, window_days=window_days)
    # niters=1 with a huge sigma keeps the loop but rejects nothing.
    naive, _ = detrend.savgol_flatten(lc, window_days=window_days, niters=1, sigma=1e9)
    print(f'{window_days:10.2f} {days_to_cadences(lc, window_days):9d} '
          f'{measured_depth(clipped):8.0f} ppm {measured_depth(naive):10.0f} ppm')

Read the right-hand column first: **without outlier rejection the transit is annihilated.** By the time the window reaches the transit duration, an 8,950 ppm signal reads as ~70 ppm — the filter has decided the transit *is* the trend and divided it out. That is the failure section 1 warned about, and it is severe.

Now the left column: with clipping on, the same window costs only ~5% of the depth. lightkurve's `flatten()` **iteratively sigma-clips before fitting the trend** (`niters=3, sigma=3`), and in-transit points are outliers to a smooth trend, so they're excluded from the very fit that would have absorbed them. The library has been quietly protecting you this whole time.

Two lessons, and the second matters more:

1. The textbook failure mode is real and total, not a rounding error.
2. **You were defended by a default you didn't choose.** That defence is weaker for long or shallow transits, where in-transit points are less outlying — and it doesn't exist at all in a filter you write yourself. Picking a window well above the transit duration is still the actual fix; clipping is a safety net, not a strategy.

### A tempting conclusion that is wrong

It is natural to read the above as "use a *robust* filter instead" — wotan's Tukey biweight down-weights outliers rather than clipping them, so surely it handles this better. Measured at matched windows on this curve, it does not:

| window | savgol (clipped) | wotan biweight |
|---|---|---|
| 18.8 d | 8,731 ppm | 8,729 ppm |
| 2.0 d | 8,741 ppm | 8,751 ppm |
| 0.5 d | 8,645 ppm | 8,717 ppm |
| 0.25 d | 8,555 ppm | 8,131 ppm |
| 0.13 d | 8,270 ppm | **354 ppm** |

At any sensible window they are equivalent; at aggressive ones lightkurve wins outright. The two robustness strategies fail differently. Sigma-clipping compares each point to a trend fitted over the *whole curve*, so in-transit points look like outliers and are excluded from the fit. The biweight is **local**: inside a 0.13-day window sitting in a 0.13-day transit, the in-transit points are the majority, so they define the local centre and are never down-weighted.

**Local robustness cannot protect a feature that fills the window.** So don't pick an estimator hoping it will rescue a bad window — pick the window, and either estimator works. (wotan still earns its place, for windows in real time rather than cadence counts, and for its menu of trend models — just not for this.)

A clean U-shaped dip at phase 0 = the period is real. That's a detection.

## Recap
- Real curves need **detrending**; a Savitzky–Golay filter divides out slow trends — but keep the window well **longer than a transit**, and know whether your library wants **days or cadences**.
- **BLS** (box-shaped) is for transits; **Lomb–Scargle** (sinusoidal) is for smooth variability.
- A periodogram peak is a *hypothesis*; **folding** confirms it.
- A recovered period is still only a *candidate*. Notebook 04 covers the checks that separate planets from impostors.

**Next:**
- **[`04_false_positives_and_noise.ipynb`](04_false_positives_and_noise.ipynb)** — why most signals like this one are *not* planets.
- **[`05_bls_vs_tls.ipynb`](05_bls_vs_tls.ipynb)** — BLS searches with a *box*; TLS searches with a real limb-darkened transit shape. Same star, same curve, sharper peak.
- **[`kepler8b_transit_recovery.ipynb`](kepler8b_transit_recovery.ipynb)** — *capstone*: this same pipeline, validated against the published values, including a cautionary bug (low-outlier clipping eating the transit) that shows why we check our answers.

## Learning resources
- 📗 [Lightkurve: removing systematics / flattening](https://docs.lightkurve.org/tutorials/2-creating-light-curves/2-3-removing-systematics.html)
- 📗 [Lightkurve: identifying transiting planets with BLS](https://docs.lightkurve.org/tutorials/3-science-examples/exoplanets-identifying-transiting-planet-signals.html)
- 📘 [Astropy: Box Least Squares](https://docs.astropy.org/en/stable/timeseries/bls.html)
- 📗 [wotan: detrending methods compared](https://github.com/hippke/wotan) — and [Hippke et al. (2019)](https://arxiv.org/abs/1906.00966) on why robust filters preserve transits
- 🌍 [Lomb–Scargle periodogram](https://en.wikipedia.org/wiki/Lomb%E2%80%93Scargle_periodogram)
- 📄 [VanderPlas (2018), *Understanding the Lomb–Scargle Periodogram*](https://arxiv.org/abs/1703.09824) — excellent deep dive if you like primary sources
- 📄 [Kovács, Zucker & Mazeh (2002), the original BLS paper](https://arxiv.org/abs/astro-ph/0206099)